# INT-01 — End-to-End Integration Tests

**Ticket:** INT-01  
**Purpose:** Validate the full Bronze → Silver → Gold → Dashboard pipeline.

Runs **123 pytest tests** across 4 test modules:

| Module | Coverage |
|--------|----------|
| `test_int01_bronze.py` | Table exists, schema, min row count |
| `test_int01_silver.py` | I-03 structure, I-04 rules, I-05 derived, I-06 referential integrity |
| `test_int01_gold.py` | Fact schema, grain uniqueness, metrics, dims, dashboard readiness |
| `test_int01_pipeline.py` | Cross-layer row count & revenue reconciliation |

> **Note:** Tests must run in-process via `pytest.main()` so the active Spark Connect session is available.

---

In [0]:
%pip install pytest -q
dbutils.library.restartPython()

In [0]:
import sys
import os

# Prevent __pycache__ writes in workspace directories
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

# Ensure project root is on sys.path for src imports
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

# Force-reload src modules to pick up latest code
import importlib
import src.constants, src.transforms, src.validators
importlib.reload(src.constants)
importlib.reload(src.transforms)
importlib.reload(src.validators)

import pytest

exit_code = pytest.main([
    "tests/test_int01_bronze.py",
    "tests/test_int01_silver.py",
    "tests/test_int01_gold.py",
    "tests/test_int01_pipeline.py",
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
])

# Fail the notebook if any tests failed
assert exit_code == 0, f"INT-01 tests failed with exit code {exit_code}"